# 🎬 Affiliate Video Bot v4.0 — Google Colab Worker

> **Hướng dẫn:** Nhấn **Runtime → Run all** (Ctrl+F9) để chạy tất cả.
> Lần đầu cần ~20-30 phút (tải models). Lần sau chỉ ~3-5 phút.
>
> Khi Cell 5 in ra `✅ COLAB WORKER SẴN SÀNG`, Telegram sẽ tự báo bạn!

---
| Bước | Cell | Thời gian |
|------|------|----------|
| 1 | Kết nối Drive & Clone repo | ~30 giây |
| 2 | Cài thư viện AI | ~10 phút |
| 3 | Tải/khôi phục AI Models | ~3 phút (có Drive cache) / ~30 phút (lần đầu) |
| 4 | Cấu hình biến môi trường | vài giây |
| 5 | Khởi động Worker | ~30 giây |

In [ ]:
#@title ⚙️ Cấu hình — Điền thông tin của bạn vào đây { display-mode: "form" }

GITHUB_REPO = "https://github.com/YOUR_USERNAME/affiliate-video-bot.git" #@param {type:"string"}
# ^ Thay YOUR_USERNAME bằng GitHub username của bạn

USE_DRIVE_CACHE = True  #@param {type:"boolean"}
# True = dùng models đã lưu trong Drive (nhanh hơn)
# False = tải models mới từ HuggingFace (lần đầu)

DRIVE_MODELS_PATH = "/content/drive/MyDrive/AffiliateBot/models"  #@param {type:"string"}
# Đường dẫn thư mục models trên Google Drive của bạn

print("✅ Cấu hình đã lưu!")
print(f"   GitHub repo: {GITHUB_REPO}")
print(f"   Drive cache: {'Bật' if USE_DRIVE_CACHE else 'Tắt'}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Kết nối Google Drive & Clone repo từ GitHub
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path
from google.colab import drive

print("📂 Kết nối Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("✅ Drive đã kết nối!")

REPO_DIR = Path("/content/affiliate-video-bot")

if REPO_DIR.exists():
    print("🔄 Cập nhật code mới nhất từ GitHub...")
    result = subprocess.run(["git", "-C", str(REPO_DIR), "pull"],
                            capture_output=True, text=True)
    print(result.stdout or "Đã cập nhật!")
else:
    print(f"📥 Clone repo từ GitHub: {GITHUB_REPO}")
    if "YOUR_USERNAME" in GITHUB_REPO:
        raise ValueError("⚠️ Chưa đổi YOUR_USERNAME! Quay lại Cell Cấu hình phía trên.")
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO, str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Clone thất bại:\n{result.stderr}")
    print("✅ Clone repo xong!")

# Thêm repo vào Python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"\n✅ CELL 1 XONG! Repo tại: {REPO_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — Cài thư viện AI (~10 phút, chỉ lần đầu)
# ══════════════════════════════════════════════════════════════
import subprocess, sys

def run(cmd, desc=""):
    if desc: print(f"⏳ {desc}...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"⚠️  Lỗi: {r.stderr[-500:]}")
    else:
        print(f"   ✓ Xong!")
    return r

# 1. Kiểm tra GPU
import torch
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM: {vram:.1f} GB")
else:
    print("⚠️  Không tìm thấy GPU! Vào Runtime → Change runtime type → T4 GPU")

# 2. PyTorch + CUDA (thường đã có sẵn trong Colab)
run([sys.executable, "-m", "pip", "install",
     "torch==2.4.0", "torchvision",
     "--index-url", "https://download.pytorch.org/whl/cu121", "-q"],
    "Kiểm tra/cài PyTorch CUDA")

# 3. Diffusers từ source (bắt buộc cho Wan2.1 >= 0.33)
run([sys.executable, "-m", "pip", "install",
     "git+https://github.com/huggingface/diffusers", "-q"],
    "Cài Diffusers từ source")

# 4. Thư viện từ requirements_colab.txt
import os
req_file = "/content/affiliate-video-bot/requirements_colab.txt"
run([sys.executable, "-m", "pip", "install", "-r", req_file,
     "--ignore-installed", "torch", "torchvision", "-q"],
    "Cài thư viện từ requirements_colab.txt")

# 5. FFmpeg
run(["apt-get", "install", "-y", "ffmpeg", "-q"], "Cài FFmpeg")

print("\n✅ CELL 2 XONG! Tất cả thư viện đã cài xong.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — Tải / Khôi phục AI Models
# Lần đầu: ~30 phút | Có Drive cache: ~3-5 phút
# ══════════════════════════════════════════════════════════════
import shutil, os, urllib.request
from pathlib import Path
from huggingface_hub import snapshot_download

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(exist_ok=True)

# ── Khôi phục từ Drive cache (nhanh hơn tải lại) ──────────────
if USE_DRIVE_CACHE and Path(DRIVE_MODELS_PATH).exists():
    print("📂 Tìm thấy models trong Drive! Đang khôi phục...")
    print("   (Nhanh hơn nhiều so với tải lại từ HuggingFace)")
    shutil.copytree(DRIVE_MODELS_PATH, str(MODELS_DIR), dirs_exist_ok=True)
    print("✅ Khôi phục models từ Drive xong!")

else:
    # ── Tải mới từ HuggingFace ───────────────────────────────────
    print("📥 Tải models từ HuggingFace (lần đầu, ~30 phút)...")
    print("💡 Tip: Sau khi tải xong, nhớ lưu vào Drive để lần sau nhanh hơn!")

    # Wan2.1 I2V 480P (~13GB)
    wan_dir = MODELS_DIR / "wan21"
    if not wan_dir.exists() or not any(wan_dir.iterdir()):
        print("\n⏳ Tải Wan2.1-I2V-14B-480P (~13GB)... ~20 phút")
        snapshot_download(
            repo_id="Wan-AI/Wan2.1-I2V-14B-480P-Diffusers",
            local_dir=str(wan_dir),
            ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
        )
        print("   ✓ Wan2.1 xong!")
    else:
        print("   ✓ Wan2.1 đã có sẵn, bỏ qua tải.")

    # Real-ESRGAN (~65MB)
    esrgan_dir = MODELS_DIR / "realesrgan"
    esrgan_dir.mkdir(exist_ok=True)
    esrgan_file = esrgan_dir / "RealESRGAN_x4plus.pth"
    if not esrgan_file.exists():
        print("\n⏳ Tải Real-ESRGAN model (~65MB)...")
        urllib.request.urlretrieve(
            "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
            str(esrgan_file),
            reporthook=lambda b, bs, t: print(f"   {min(b*bs, t)/1e6:.1f}/{t/1e6:.1f} MB", end="\r") if t > 0 else None
        )
        print("\n   ✓ Real-ESRGAN xong!")
    else:
        print("   ✓ Real-ESRGAN đã có sẵn, bỏ qua tải.")

    # ── Lưu vào Drive để dùng lại lần sau ────────────────────────
    print("\n☁️  Đang lưu models vào Drive để dùng lại lần sau...")
    Path(DRIVE_MODELS_PATH).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(MODELS_DIR), DRIVE_MODELS_PATH, dirs_exist_ok=True)
    print(f"   ✓ Đã lưu vào: {DRIVE_MODELS_PATH}")

# ── Kiểm tra ──────────────────────────────────────────────────
total_gb = sum(f.stat().st_size for f in MODELS_DIR.rglob("*") if f.is_file()) / 1e9
print(f"\n📊 Tổng kích thước models: {total_gb:.1f} GB")
print(f"✅ CELL 3 XONG! Models sẵn sàng tại: {MODELS_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — Set biến môi trường từ Colab Secrets
# Đảm bảo đã thêm Secrets trong panel bên trái (icon 🔑)
# ══════════════════════════════════════════════════════════════
import os
from google.colab import userdata

REQUIRED_SECRETS = [
    "TELEGRAM_BOT_TOKEN",
    "DISPATCHER_URL",
    "NGROK_AUTH_TOKEN",
]

OPTIONAL_SECRETS = [
    "GDRIVE_CREDENTIALS_JSON",
    "GDRIVE_ROOT_FOLDER_ID",
]

print("🔑 Đang load Secrets...")
errors = []

for key in REQUIRED_SECRETS:
    try:
        val = userdata.get(key)
        os.environ[key] = val
        print(f"   ✅ {key}: {'*' * min(len(val), 8)}...")
    except Exception:
        errors.append(key)
        print(f"   ❌ {key}: CHƯA SET!")

for key in OPTIONAL_SECRETS:
    try:
        val = userdata.get(key)
        os.environ[key] = val
        print(f"   ✅ {key}: có ({len(val)} ký tự)")
    except Exception:
        print(f"   ⚠️  {key}: chưa set (Drive upload sẽ bị bỏ qua)")

# Biến cố định
os.environ["MODELS_DIR"]          = str(MODELS_DIR)
os.environ["VIDEO_ENGINE"]        = "auto"
os.environ["REALESRGAN_SCALE"]    = "2"
os.environ["WORKER_PORT"]         = "8000"

if errors:
    msg = (
        f"❌ Thiếu {len(errors)} Secret bắt buộc: {', '.join(errors)}\n"
        "\n👉 Cách thêm Secret trong Colab:\n"
        "   1. Click icon 🔑 ở sidebar trái\n"
        "   2. Nhấn + Add new secret\n"
        "   3. Nhập Name và Value\n"
        "   4. Chạy lại cell này"
    )
    raise ValueError(msg)

print(f"\n✅ CELL 4 XONG! Tất cả Secrets đã set.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — Khởi động Colab Worker
# Khi thấy "✅ COLAB WORKER SẴN SÀNG"
# → Telegram sẽ tự báo "🟢 Colab đã sẵn sàng!"
# ══════════════════════════════════════════════════════════════
import sys, asyncio

# Đảm bảo repo trong path
repo = "/content/affiliate-video-bot"
if repo not in sys.path:
    sys.path.insert(0, repo)

# Xóa cache import cũ nếu có
for mod in ["colab_worker"]:
    if mod in sys.modules:
        del sys.modules[mod]

print("🚀 Đang khởi động Colab Worker...")
print(f"   Dispatcher URL: {os.environ.get('DISPATCHER_URL', '(chưa set)')}")
print(f"   Bot token: {'✅ có' if os.environ.get('TELEGRAM_BOT_TOKEN') else '❌ thiếu'}")
print(f"   ngrok token: {'✅ có' if os.environ.get('NGROK_AUTH_TOKEN') else '❌ thiếu'}")
print()

from colab_worker import start_worker

# Chạy worker (blocking — cell này chạy mãi cho đến khi session kết thúc)
asyncio.run(start_worker())

In [ ]:
#@title 🛠️ Công cụ tiện ích (chạy khi cần) { display-mode: "form" }

TOOL = "Xem trạng thái GPU" #@param ["Xem trạng thái GPU", "Kiểm tra kết nối Dispatcher", "Xoá cache GPU", "Xem dung lượng ổ cứng", "Lưu models vào Drive ngay"]

import os, torch
from pathlib import Path

if TOOL == "Xem trạng thái GPU":
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        total = props.total_memory / 1e9
        used  = torch.cuda.memory_allocated(0) / 1e9
        free  = total - used
        print(f"🎮 GPU: {props.name}")
        print(f"   Tổng VRAM: {total:.1f} GB")
        print(f"   Đang dùng: {used:.2f} GB")
        print(f"   Còn trống: {free:.2f} GB")
    else:
        print("❌ Không có GPU!")

elif TOOL == "Kiểm tra kết nối Dispatcher":
    import httpx, asyncio
    url = os.environ.get("DISPATCHER_URL", "")
    if not url:
        print("❌ DISPATCHER_URL chưa set")
    else:
        async def check():
            try:
                async with httpx.AsyncClient(timeout=10) as c:
                    r = await c.get(f"{url}/health")
                    data = r.json()
                    print(f"✅ Dispatcher đang chạy: {url}")
                    print(f"   Colab registered: {data.get('colab_connected')}")
                    print(f"   Last ping: {data.get('last_ping')}")
            except Exception as e:
                print(f"❌ Không kết nối được: {e}")
        asyncio.run(check())

elif TOOL == "Xoá cache GPU":
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("✅ Đã xoá cache GPU")

elif TOOL == "Xem dung lượng ổ cứng":
    import shutil
    total, used, free = shutil.disk_usage("/content")
    print(f"💾 Ổ cứng /content:")
    print(f"   Tổng: {total/1e9:.1f} GB")
    print(f"   Dùng: {used/1e9:.1f} GB")
    print(f"   Trống: {free/1e9:.1f} GB")

elif TOOL == "Lưu models vào Drive ngay":
    import shutil
    src = Path("/content/models")
    dst = Path(DRIVE_MODELS_PATH)
    if not src.exists():
        print("❌ Chưa có thư mục /content/models")
    else:
        print(f"☁️  Đang lưu {src} → {dst}...")
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        size = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e9
        print(f"✅ Đã lưu vào Drive! ({size:.1f} GB)")